# Phase 8 – ETL Pipeline Automation

**Project:** Automated Data Quality & ETL Framework  
**Dataset:** Online Retail

## Objective

This notebook automates the complete ETL workflow:

1. Extract the raw Excel dataset.
2. Standardize column names.
3. validate data quality issues.
4. Clean invalid and unwanted records.
5. Create analytical features.
6. Save the final transformed dataset.
7. Generate a pipeline execution summary.

Running the final cell executes the entire pipeline from beginning to end.


## 1. Import Libraries

- `pandas` is used for data processing.
- `os` is used to build operating-system-safe paths.
- `datetime` records the pipeline execution timestamp.


In [2]:
import os
from datetime import datetime

import pandas as pd


## 2. Configure Project Paths

All paths are defined in one place so the pipeline is easier to maintain.

Update `BASE_DIR` only when the project folder is moved to another location.


In [3]:
BASE_DIR = r"C:\Users\chatu\OneDrive\Desktop\Automated-Data-Quality-ETL-Framework"

RAW_FILE_PATH = os.path.join(
    BASE_DIR,
    "data",
    "raw",
    "Online Retail.xlsx"
)

OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
REPORT_DIR = os.path.join(BASE_DIR, "reports", "automation")

FINAL_OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "feature_engineered_online_retail.csv"
)

PIPELINE_SUMMARY_FILE = os.path.join(
    REPORT_DIR,
    "phase_8_etl_pipeline_summary.csv"
)


## 3. Create Required Folders

The pipeline creates the output and automation-report folders when they do not already exist.


In [4]:
def create_required_folders():
    """Create directories required for pipeline outputs and reports."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(REPORT_DIR, exist_ok=True)


## 4. Extract Raw Data

The extraction function checks whether the source file exists before reading it.  
A clear `FileNotFoundError` is raised when the configured file path is incorrect.


In [5]:
def extract_data(file_path):
    """Read the raw Online Retail Excel file into a DataFrame."""
    print("STEP 1  Data Extraction")
    print("-" * 65)
    print("Extracting raw data...")

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Raw input file was not found at: {file_path}"
        )

    dataframe = pd.read_excel(file_path)

    print(
        "Completed: Raw data loaded successfully "
        f"({dataframe.shape[0]:,} rows, {dataframe.shape[1]} columns)"
    )
    print()

    return dataframe


## 5. Standardize and Verify Column Names

The source dataset uses names such as `InvoiceNo` and `CustomerID`.  
These are converted into consistent `snake_case` names such as `invoice_no` and `customer_id`.

The function also verifies that all required columns are present before the pipeline continues.


In [6]:
EXPECTED_COLUMNS = {
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country",
}


def standardize_column_names(dataframe):
    """Rename source columns and confirm that required fields exist."""
    print("STEP 2  Column Standardization")
    print("-" * 65)
    print("Standardizing column names...")

    column_mapping = {
        "InvoiceNo": "invoice_no",
        "StockCode": "stock_code",
        "Description": "description",
        "Quantity": "quantity",
        "InvoiceDate": "invoice_date",
        "UnitPrice": "unit_price",
        "CustomerID": "customer_id",
        "Country": "country",
    }

    standardized_df = dataframe.rename(columns=column_mapping).copy()

    standardized_df.columns = (
        standardized_df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

    missing_columns = EXPECTED_COLUMNS.difference(standardized_df.columns)

    if missing_columns:
        raise KeyError(
            "The dataset is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    print("Completed: Column names standardized")
    print()

    return standardized_df


## 6. Validate Raw Data

Validation records the main data-quality issues before cleaning:

- Duplicate rows
- Missing key fields
- Cancelled invoices
- Non-positive quantity
- Non-positive unit price

These values are later written to the pipeline summary report.


In [7]:
def validate_data(dataframe):
    """Calculate data-quality metrics before cleaning begins."""
    print("STEP 3  Data Validation")
    print("-" * 65)
    print("Running data validation checks...")

    validation_results = {
        "total_rows": len(dataframe),
        "total_columns": dataframe.shape[1],
        "duplicate_rows": int(dataframe.duplicated().sum()),
        "missing_invoice_no": int(dataframe["invoice_no"].isna().sum()),
        "missing_stock_code": int(dataframe["stock_code"].isna().sum()),
        "missing_description": int(dataframe["description"].isna().sum()),
        "missing_customer_id": int(dataframe["customer_id"].isna().sum()),
        "negative_or_zero_quantity": int(
            (dataframe["quantity"] <= 0).sum()
        ),
        "negative_or_zero_unit_price": int(
            (dataframe["unit_price"] <= 0).sum()
        ),
        "cancelled_invoices": int(
            dataframe["invoice_no"]
            .astype(str)
            .str.startswith("C", na=False)
            .sum()
        ),
    }

    print("Completed: Data validation checks finished")
    print()

    return validation_results


## 7. Clean the Data

The cleaning stage:

1. Removes exact duplicate rows.
2. Removes cancelled invoices.
3. Removes records without a customer ID.
4. Removes non-positive quantities and prices.
5. Fills missing descriptions.
6. Converts data types required by later steps.


In [8]:
def clean_data(dataframe):
    """Apply the approved cleaning rules to the raw dataset."""
    print("STEP 4  Data Cleaning")
    print("-" * 65)
    print("Cleaning data...")

    initial_rows = len(dataframe)

    cleaned_df = dataframe.drop_duplicates().copy()

    cleaned_df = cleaned_df[
        ~cleaned_df["invoice_no"]
        .astype(str)
        .str.startswith("C", na=False)
    ].copy()

    cleaned_df = cleaned_df.dropna(
        subset=["customer_id", "invoice_date"]
    ).copy()

    cleaned_df = cleaned_df[
        (cleaned_df["quantity"] > 0)
        & (cleaned_df["unit_price"] > 0)
    ].copy()

    cleaned_df["description"] = (
        cleaned_df["description"]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

    cleaned_df["customer_id"] = cleaned_df["customer_id"].astype(int)

    cleaned_df["invoice_date"] = pd.to_datetime(
        cleaned_df["invoice_date"],
        errors="raise"
    )

    cleaned_df = cleaned_df.reset_index(drop=True)

    removed_rows = initial_rows - len(cleaned_df)

    print(
        "Completed: Data cleaning finished "
        f"({removed_rows:,} rows removed, {len(cleaned_df):,} rows retained)"
    )
    print()

    return cleaned_df, removed_rows


## 8. Feature Engineering

The feature-engineering stage creates:

- Transaction revenue
- Year, month, day, hour, and quarter attributes
- A monthly reporting key
- Quantity bands
- Revenue-value bands

These columns make the output ready for SQL analysis and reporting.


In [9]:
def feature_engineering(dataframe):
    """Create analytical features from cleaned transaction data."""
    print("STEP 5  Feature Engineering")
    print("-" * 65)
    print("Creating analytical features...")

    feature_df = dataframe.copy()

    feature_df["total_amount"] = (
        feature_df["quantity"] * feature_df["unit_price"]
    ).round(2)

    feature_df["invoice_year"] = feature_df["invoice_date"].dt.year
    feature_df["invoice_month"] = feature_df["invoice_date"].dt.month
    feature_df["invoice_month_name"] = (
        feature_df["invoice_date"].dt.month_name()
    )
    feature_df["invoice_day"] = feature_df["invoice_date"].dt.day
    feature_df["invoice_day_name"] = (
        feature_df["invoice_date"].dt.day_name()
    )
    feature_df["invoice_hour"] = feature_df["invoice_date"].dt.hour
    feature_df["invoice_quarter"] = feature_df["invoice_date"].dt.quarter

    feature_df["year_month"] = (
        feature_df["invoice_date"]
        .dt.to_period("M")
        .astype(str)
    )

    feature_df["quantity_category"] = pd.cut(
        feature_df["quantity"],
        bins=[0, 5, 20, 100, float("inf")],
        labels=["Low", "Medium", "High", "Bulk"],
        include_lowest=True,
    )

    feature_df["revenue_category"] = pd.cut(
        feature_df["total_amount"],
        bins=[0, 20, 100, 500, float("inf")],
        labels=[
            "Low Value",
            "Medium Value",
            "High Value",
            "Premium Value",
        ],
        include_lowest=True,
    )

    print(
        "Completed: Feature-engineered columns created "
        f"({feature_df.shape[1]} final columns)"
    )
    print()

    return feature_df


## 9. Save the Final Dataset

The completed feature-engineered dataset is saved as a CSV file in the project's `outputs` folder.


In [10]:
def save_output(dataframe, output_file):
    """Save the final feature-engineered dataset as a CSV file."""
    print("STEP 6  Export Processed Dataset")
    print("-" * 65)

    dataframe.to_csv(output_file, index=False)

    print("Completed: Feature-engineered dataset exported")
    print("Output: outputs/feature_engineered_online_retail.csv")
    print()


## 10. Save the Pipeline Summary

The summary report records the execution date, row counts, final column count, and the data-quality issues found before cleaning.


In [11]:
def save_pipeline_summary(
    raw_rows,
    final_rows,
    removed_rows,
    final_columns,
    validation_results,
):
    """Save one-row pipeline execution and validation summary."""
    print("STEP 7  Generate Pipeline Report")
    print("-" * 65)

    summary_data = {
        "execution_date": [
            datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ],
        "pipeline_status": ["Success"],
        "raw_rows": [raw_rows],
        "final_rows": [final_rows],
        "removed_rows": [removed_rows],
        "final_columns": [final_columns],
        "duplicate_rows_found": [
            validation_results["duplicate_rows"]
        ],
        "missing_customer_id_found": [
            validation_results["missing_customer_id"]
        ],
        "cancelled_invoices_found": [
            validation_results["cancelled_invoices"]
        ],
        "invalid_quantity_rows_found": [
            validation_results["negative_or_zero_quantity"]
        ],
        "invalid_unit_price_rows_found": [
            validation_results["negative_or_zero_unit_price"]
        ],
    }

    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv(PIPELINE_SUMMARY_FILE, index=False)

    print("Completed: ETL pipeline summary report generated")
    print("Report: reports/automation/phase_8_etl_pipeline_summary.csv")
    print()


## 11. Orchestrate the Complete ETL Pipeline

This function controls the order of execution.  
Each stage receives the output of the previous stage, creating one complete automated workflow.


In [12]:
def run_etl_pipeline():
    """Execute the complete automated ETL workflow."""
    print("=" * 65)
    print("        AUTOMATED DATA QUALITY ETL FRAMEWORK")
    print("=" * 65)
    print()

    create_required_folders()

    raw_df = extract_data(RAW_FILE_PATH)
    raw_rows = len(raw_df)

    standardized_df = standardize_column_names(raw_df)

    validation_results = validate_data(standardized_df)

    cleaned_df, removed_rows = clean_data(standardized_df)

    final_df = feature_engineering(cleaned_df)

    save_output(final_df, FINAL_OUTPUT_FILE)

    save_pipeline_summary(
        raw_rows=raw_rows,
        final_rows=len(final_df),
        removed_rows=removed_rows,
        final_columns=final_df.shape[1],
        validation_results=validation_results,
    )

    print("=" * 65)
    print("        ETL PIPELINE COMPLETED SUCCESSFULLY")
    print("=" * 65)

    return final_df, validation_results


## 12. Run the Pipeline

Execute this cell after running all preceding cells.

The returned objects are:

- `final_df`: the completed feature-engineered dataset
- `validation_results`: the data-quality metrics recorded before cleaning


In [13]:
if __name__ == "__main__":
    final_df, validation_results = run_etl_pipeline()


        AUTOMATED DATA QUALITY ETL FRAMEWORK

STEP 1  Data Extraction
-----------------------------------------------------------------
Extracting raw data...
Completed: Raw data loaded successfully (541,909 rows, 8 columns)

STEP 2  Column Standardization
-----------------------------------------------------------------
Standardizing column names...
Completed: Column names standardized

STEP 3  Data Validation
-----------------------------------------------------------------
Running data validation checks...
Completed: Data validation checks finished

STEP 4  Data Cleaning
-----------------------------------------------------------------
Cleaning data...
Completed: Data cleaning finished (149,217 rows removed, 392,692 rows retained)

STEP 5  Feature Engineering
-----------------------------------------------------------------
Creating analytical features...
Completed: Feature-engineered columns created (19 final columns)

STEP 6  Export Processed Dataset
------------------------------

## Expected Outputs

After successful execution, the notebook creates:

```text
outputs/
└── feature_engineered_online_retail.csv

reports/
└── automation/
    └── phase_8_etl_pipeline_summary.csv
```

## Pipeline Flow

```text
Raw Excel file
      ↓
Extraction
      ↓
Column standardization
      ↓
Data validation
      ↓
Data cleaning
      ↓
Feature engineering
      ↓
Final CSV and summary report
```
